In [ ]:
# =============================================
# A* PATH PLANNING
# =============================================
def astar(cost_map, start, goal):
    rows, cols = cost_map.shape
    open_set   = []
    heapq.heappush(open_set, (0, start))

    came_from  = {}
    g_score    = {start: 0}
    f_score    = {start: abs(start[0]-goal[0]) + abs(start[1]-goal[1])}

    while open_set:
        _, current = heapq.heappop(open_set)

        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
            nr, nc = current[0]+dr, current[1]+dc
            if 0 <= nr < rows and 0 <= nc < cols:
                neighbor   = (nr, nc)
                tentative_g = g_score[current] + cost_map[nr, nc]
                if tentative_g < g_score.get(neighbor, float('inf')):
                    came_from[neighbor] = current
                    g_score[neighbor]   = tentative_g
                    h = abs(nr-goal[0]) + abs(nc-goal[1])
                    f_score[neighbor]   = tentative_g + h
                    heapq.heappush(open_set, (f_score[neighbor], neighbor))

    return None

# Set start and goal
start = (0, 0)
goal  = (rows-1, cols-1)

path = astar(cost_grid, start, goal)
print(f"\nPath found: {len(path)} steps" if path else "\nNo path found!")


In [ ]:
# =============================================
# SMOOTH PATH
# =============================================
def smooth_path(path, sigma=1.5):
    if path is None or len(path) < 3:
        return path
    path_arr    = np.array(path, dtype=float)
    smooth_rows = gaussian_filter(path_arr[:, 0], sigma=sigma)
    smooth_cols = gaussian_filter(path_arr[:, 1], sigma=sigma)
    return list(zip(smooth_rows, smooth_cols))

smooth = smooth_path(path)


In [ ]:
# =============================================
# PATH NAVIGABILITY
# =============================================
path_nav = float(np.mean([navigability_grid[r, c] for r, c in path]) * 100)
print(f"Planned Path Navigability: {path_nav:.2f}%")


In [ ]:
# =============================================
# VISUALIZE FINAL PATH
# =============================================
scale  = PATCH_SIZE
path_img   = np.array([(c*scale + scale//2, r*scale + scale//2) for r, c in path])
smooth_img = np.array([(c*scale + scale//2, r*scale + scale//2) for r, c in smooth])

# Overlay grid on image
overlay = img_rgb.copy()
for r in range(rows):
    for c in range(cols):
        cost  = cost_grid[r, c]
        color = (int(cost*255), int((1-cost)*255), 0)
        pt1   = (c*scale, r*scale)
        pt2   = ((c+1)*scale, (r+1)*scale)
        cv2.rectangle(overlay, pt1, pt2, color, -1)

overlay = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

plt.figure(figsize=(12, 8))
plt.imshow(overlay)
plt.plot(path_img[:, 0],   path_img[:, 1],   'b--', linewidth=1.5, label="A* Grid Path")
plt.plot(smooth_img[:, 0], smooth_img[:, 1], color="orange", linewidth=2.5, label="Smooth Rover Path")
plt.scatter(path_img[0, 0],  path_img[0, 1],  s=120, marker='o', color="blue", label="Start")
plt.scatter(path_img[-1, 0], path_img[-1, 1], s=120, marker='X', color="orange", label="Goal")
plt.title("Smoothed Rover Path using Safety + Navigability")
plt.legend()
plt.axis("off")
plt.show()


In [ ]:
# =============================================
# DECISION OUTPUT
# =============================================
dangerous_cells = [(r, c) for r in range(rows) for c in range(cols) if cost_grid[r, c] > 0.8]
risky_cells     = [(r, c) for r in range(rows) for c in range(cols) if 0.6 < cost_grid[r, c] <= 0.8]
difficult_cells = [(r, c) for r in range(rows) for c in range(cols) if 0.4 < cost_grid[r, c] <= 0.6]

if len(dangerous_cells) > 0:
    decision = "REROUTE"
    reason   = "Dangerous terrain ahead"
elif len(difficult_cells) > 0 or len(risky_cells) > 0:
    decision = "SLOW_DOWN"
    reason   = "Difficult or risky terrain ahead"
else:
    decision = "MOVE_FORWARD"
    reason   = "Safe path ahead"

print("\nFINAL DECISION:")
print("Decision:", decision)
print("Reason  :", reason)

In [ ]:
# =============================================
# GRAD-CAM (FIXED)
# =============================================
patch_index = 0
patch_rgb   = patches[patch_index]

input_patch = tf.convert_to_tensor(patch_rgb, dtype=tf.float32)
input_patch = tf.image.resize(input_patch, (IMG_SIZE, IMG_SIZE))
input_patch = preprocess_input(input_patch)
input_patch = tf.expand_dims(input_patch, axis=0)

# Get layers directly from model (no nested base_model)
last_conv_layer  = model.get_layer("out_relu")      # layer 153
gap_layer        = model.get_layer("global_average_pooling2d")  # layer 154
dense_layer      = model.get_layer("dense")          # layer 155
dropout_layer    = model.get_layer("dropout")        # layer 156
output_layer     = model.get_layer("dense_1")        # layer 157

# Build conv model up to last conv layer
last_conv_model = tf.keras.models.Model(
    inputs=model.input,
    outputs=last_conv_layer.output
)

# Grad-CAM
with tf.GradientTape() as tape:
    conv_outputs = last_conv_model(input_patch)
    tape.watch(conv_outputs)
    x           = gap_layer(conv_outputs)
    x           = dense_layer(x)
    x           = dropout_layer(x, training=False)
    predictions = output_layer(x)

    predicted_class = tf.argmax(predictions[0])
    class_channel   = predictions[:, predicted_class]

grads        = tape.gradient(class_channel, conv_outputs)
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
conv_out     = conv_outputs[0]
heatmap      = conv_out @ pooled_grads[..., tf.newaxis]
heatmap      = tf.squeeze(heatmap).numpy()
heatmap      = np.maximum(heatmap, 0)
heatmap      = heatmap / np.max(heatmap) if np.max(heatmap) != 0 else heatmap

heatmap_resized = cv2.resize(heatmap, (patch_rgb.shape[1], patch_rgb.shape[0]))
pred_class_name = CLASS_NAMES[int(predicted_class)]

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(patch_rgb)
plt.title("Original Patch")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(heatmap_resized, cmap="jet")
plt.title("Grad-CAM Heatmap")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(patch_rgb)
plt.imshow(heatmap_resized, cmap="jet", alpha=0.4)
plt.title(f"Overlay ({pred_class_name})")
plt.axis("off")

plt.tight_layout()
plt.show()

print(f"\nPredicted terrain for patch {patch_index}: {pred_class_name}")